## Cleaning criteria
Chicago townships 70-77, arm's-length (all `sale_filter_*` == False), no multi-parcel sales, single-card + single-family only.

In [ ]:
import pandas as pd

DATA_DIR = 

SALES_PATH     = 
IMP_PATH       = 
UNIVERSE_PATH  = 
PROXIMITY_PATH = 
OUT_PATH       = 

SALES_COLS = [
    "pin", "year", "township_code", "neighborhood_code", "class",
    "sale_date", "sale_price", "is_multisale",
    "sale_filter_same_sale_within_365", "sale_filter_less_than_10k", "sale_filter_deed_type",
]

IMP_COLS = [
    "pin", "tax_year", "card_num", "pin_is_multicard",
    "year_built", "building_sqft", "land_sqft",
    "num_bedrooms", "num_rooms", "num_full_baths", "num_half_baths",
    "num_fireplaces", "type_of_residence", "construction_quality",
    "repair_condition", "basement_type", "basement_finish",
    "attic_finish", "central_air", "central_heating", "garage_size",
    "ext_wall_material", "single_v_multi_family",
]

UNIVERSE_COLS = [
    "pin", "pin10", "tax_year",
    "zip_code", "longitude", "latitude", "centroid_x_crs_3435", "centroid_y_crs_3435",
    "chicago_community_area_num", "flood_fema_sfha", "flood_fs_factor", "airport_noise_dnl",
    "school_elementary_district_name", "school_secondary_district_name",
    "cmap_walkability_total_score",
    "census_acs5_tract_geoid",
]

PROXIMITY_COLS = [
    "pin10", "year",
    "num_pin_in_half_mile", "num_bus_stop_in_half_mile",
    "num_foreclosure_in_half_mile_past_5_years", "num_foreclosure_per_1000_pin_past_5_years",
    "num_school_in_half_mile", "airport_dnl_total", "lake_michigan_dist_ft",
    "nearest_bike_trail_dist_ft", "nearest_cemetery_dist_ft", "nearest_cta_route_dist_ft",
    "nearest_cta_stop_dist_ft", "nearest_golf_course_dist_ft", "nearest_hospital_dist_ft",
    "nearest_major_road_dist_ft", "nearest_metra_route_dist_ft", "nearest_metra_stop_dist_ft",
    "nearest_new_construction_dist_ft", "nearest_park_dist_ft", "nearest_railroad_dist_ft",
    "nearest_road_arterial_dist_ft", "nearest_road_arterial_daily_traffic",
    "nearest_road_collector_dist_ft", "nearest_road_collector_daily_traffic",
    "nearest_road_highway_dist_ft", "nearest_road_highway_daily_traffic",
    "nearest_secondary_road_dist_ft", "nearest_stadium_dist_ft", "nearest_university_dist_ft",
    "nearest_vacant_land_dist_ft", "nearest_water_dist_ft",
    "nearest_neighbor_1_dist_ft", "nearest_neighbor_2_dist_ft", "nearest_neighbor_3_dist_ft",
    "nearest_new_construction_pin10", "nearest_vacant_land_pin10",
    "nearest_neighbor_1_pin10", "nearest_neighbor_2_pin10", "nearest_neighbor_3_pin10",
]

In [ ]:
def norm_pin(s, width=14):
    """Strip non-digits and zero-pad to `width` (14 for PIN, 10 for PIN10)."""
    return s.astype(str).str.replace(r"\D", "", regex=True).str.zfill(width)

def clean_num(s):
    """Parse price/area columns stored in mixed locale formats (US commas, EU comma-decimal, space-thousands)."""
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s", "", regex=True)
    hc = s.str.contains(",")
    hd = s.str.contains(r"\.", regex=True)
    s = s.mask(hc & hd, s.str.replace(",", "", regex=False))     # US: comma = thousands sep
    s = s.mask(hc & ~hd, s.str.replace(",", ".", regex=False))   # EU: comma = decimal
    s = s.str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(s.replace("", pd.NA), errors="coerce")

def is_false(s):
    """True if column value is False (works for both bool and string 'false')."""
    return s.astype(str).str.lower() == "false"

def asof(left, right, by):
    """Join right onto left using the most recent year at or before each left row."""
    return pd.merge_asof(left.sort_values("year"), right.sort_values("year"),
                         on="year", by=by, direction="backward")

def load_chicago(path, usecols, key_col, keep_keys, key_width, post=None, chunksize=400_000):
    """Stream a large CSV, normalize join key, and keep only rows in keep_keys."""
    parts = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=False):
        chunk[key_col] = norm_pin(chunk[key_col], key_width)
        chunk = chunk[chunk[key_col].isin(keep_keys)]
        if post is not None:
            chunk = post(chunk)
        parts.append(chunk)
    return pd.concat(parts, ignore_index=True)

## 1. Sales - load, clean, filter

Apply arm's-length filters, drop multi-parcel sales, keep Chicago townships 70-77, years 2021-2025.

In [3]:
sales = pd.read_csv(SALES_PATH, usecols=SALES_COLS, low_memory=False)
sales["pin"] = norm_pin(sales["pin"])
sales["sale_price"] = clean_num(sales["sale_price"])

n0 = len(sales)
arms = (is_false(sales["sale_filter_same_sale_within_365"])
        & is_false(sales["sale_filter_less_than_10k"])
        & is_false(sales["sale_filter_deed_type"]))
sales = sales[arms]
sales = sales[~(sales["is_multisale"].astype(str).str.lower() == "true")]
sales = sales[sales["sale_price"] > 0]
sales = sales[sales["township_code"].between(70, 77)]
sales = sales[sales["year"].between(2021, 2025)].copy()

chicago_pins = set(sales["pin"])
chicago_pin10s = {p[:10] for p in chicago_pins}
print(f"Sales: {n0:,} -> {len(sales):,} (arm's-length, single-parcel, Chicago, 2021-2025)")
print(f"Unique Chicago PINs: {len(chicago_pins):,}")

Sales: 2,643,013 -> 150,415 (arm's-length, single-parcel, Chicago, 2021-2025)
Unique Chicago PINs: 132,000


## 2. Improvements - chunked load (single-card, single-family, 2021+)

Stream 25.8M rows in chunks; keep Chicago + single-card + single-family + tax_year >= 2021.

In [4]:
def imp_filter(chunk):
    chunk = chunk[chunk["tax_year"] >= 2021]  # cap at 2021+; pre-HIE records excluded
    chunk = chunk[is_false(chunk["pin_is_multicard"])]
    return chunk[chunk["single_v_multi_family"] == "Single-Family"]

imp = load_chicago(IMP_PATH, IMP_COLS, "pin", chicago_pins, 14, post=imp_filter)
imp["building_sqft"] = clean_num(imp["building_sqft"])
imp["land_sqft"] = clean_num(imp["land_sqft"])
imp = imp.drop(columns=["pin_is_multicard", "card_num"])
imp = imp.rename(columns={"tax_year": "year"})

dups = imp.duplicated(["pin", "year"]).sum()
print(f"Improvements (Chicago, single-card, single-family, 2021+): {len(imp):,} rows | dup pin+year: {dups}")
if dups:
    imp = imp.drop_duplicates(["pin", "year"], keep="last")

Improvements (Chicago, single-card, single-family, 2021+): 292,084 rows | dup pin+year: 0


## 3. Join Sales -> Improvements (merge_asof, year-aware)

In [5]:
df = asof(sales, imp, by="pin")
assert len(df) == len(sales), "row count changed in sales->improvements join!"
matched = df["building_sqft"].notna().sum()
print(f"After improvements join: {len(df):,} rows | with characteristics: "
      f"{matched:,} ({100*matched/len(df):.1f}%)")

After improvements join: 150,415 rows | with characteristics: 56,630 (37.6%)


## 4. Join Parcel Universe (geography)

Adds coordinates, flood/noise risk, schools, community area, walkability, and pin10 (needed for the proximity join).

In [6]:
uni = load_chicago(UNIVERSE_PATH, UNIVERSE_COLS, "pin", chicago_pins, 14)
uni["pin10"] = norm_pin(uni["pin10"], 10)
uni = uni.rename(columns={"tax_year": "year"})
dups = uni.duplicated(["pin", "year"]).sum()
print(f"Universe (Chicago): {len(uni):,} rows | dup pin+year: {dups}")
if dups:
    uni = uni.drop_duplicates(["pin", "year"], keep="last")

n_before = len(df)
df = asof(df, uni, by="pin")
assert len(df) == n_before, "row count changed in universe join!"
print(f"After universe join: {len(df):,} rows x {df.shape[1]} cols | "
      f"pin10 present: {df['pin10'].notna().mean()*100:.1f}%")

Universe (Chicago): 656,030 rows | dup pin+year: 0


After universe join: 150,415 rows x 44 cols | pin10 present: 100.0%


## 5. Join Parcel Proximity (amenity distances, on PIN10)

In [7]:
df["pin10"] = df["pin10"].fillna(df["pin"].str[:10])
prox = load_chicago(PROXIMITY_PATH, PROXIMITY_COLS, "pin10", chicago_pin10s, 10,
                    post=lambda c: c[c["year"] >= 2019])
dups = prox.duplicated(["pin10", "year"]).sum()
print(f"Proximity (Chicago, year>=2019): {len(prox):,} rows | dup pin10+year: {dups}")
if dups:
    prox = prox.drop_duplicates(["pin10", "year"], keep="last")

n_before = len(df)
df = asof(df, prox, by="pin10")
assert len(df) == n_before, "row count changed in proximity join!"
matched = df["nearest_cta_stop_dist_ft"].notna().sum()
print(f"After proximity join: {len(df):,} rows x {df.shape[1]} cols | "
      f"with proximity: {matched:,} ({100*matched/len(df):.1f}%)")

Proximity (Chicago, year>=2019): 441,312 rows | dup pin10+year: 0


After proximity join: 150,415 rows x 82 cols | with proximity: 150,414 (100.0%)


## 6. Drop unmatched & save

Keep only sales with matched characteristics; downcast text to category; save Parquet.

In [ ]:
df = df[df["building_sqft"].notna()].copy()

# Fix locale-formatted numbers (comma-decimal, space-thousands)
num_cols = (["longitude", "latitude", "centroid_x_crs_3435", "centroid_y_crs_3435",
             "airport_noise_dnl", "flood_fs_factor", "cmap_walkability_total_score",
             "airport_dnl_total", "lake_michigan_dist_ft",
             "num_pin_in_half_mile", "num_bus_stop_in_half_mile",
             "num_foreclosure_in_half_mile_past_5_years",
             "num_foreclosure_per_1000_pin_past_5_years", "num_school_in_half_mile"]
            + [c for c in df.columns if c.endswith(("_dist_ft", "_daily_traffic"))])
for c in num_cols:
    if c in df.columns:
        df[c] = clean_num(df[c])

# Normalize nearest_*_pin10 ID columns to 10-digit strings
for c in [c for c in df.columns if c.endswith("_pin10")]:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64").astype("string").str.zfill(10)

# Downcast repeated-text columns to category
cat_cols = ["class", "type_of_residence", "construction_quality", "repair_condition",
            "basement_type", "basement_finish", "attic_finish", "ext_wall_material",
            "single_v_multi_family", "garage_size",
            "school_elementary_district_name", "school_secondary_district_name"]
for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].astype("category")

print(f"Final modeling rows: {len(df):,} x {df.shape[1]} cols")
print(f"In-memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"Numeric feature columns: {df.select_dtypes('number').shape[1]}")

df.to_parquet(OUT_PATH, index=False)
print(f"Saved -> {OUT_PATH}")


df.head()